# Amazon Bedrock Low-Level API: `InvokeModel`

*This notebook demonstrates how to get started with Amazon Bedrock using **Amazon first-party models** (Amazon Nova and Amazon Titan) via the low-level `InvokeModel` API. Adapted from [AWS Samples: 01_invoke_api.ipynb](https://github.com/aws-samples/amazon-bedrock-samples/blob/main/introduction-to-bedrock/bedrock_apis/01_invoke_api.ipynb).*

## Architecture & Context

Amazon Bedrock provides unified API access to high-performing foundation models. The `invoke_model` API call runs synchronous inference against any supported model.

![Amazon Bedrock Architecture](https://raw.githubusercontent.com/aws-samples/amazon-bedrock-samples/main/introduction-to-bedrock/bedrock_apis/assets/how-to-bedrock-api.png)


## Supported Amazon Foundation Models

This notebook focuses strictly on Amazon's native first-party foundation models:

| Model Family | Model Name | Model ID / Inference Profile | Best For |
| :--- | :--- | :--- | :--- |
| **Amazon Nova** | Nova Micro | `us.amazon.nova-micro-v1:0` | Ultra-fast, lowest cost text-only tasks |
| **Amazon Nova** | Nova Lite | `us.amazon.nova-lite-v1:0` | Fast, cost-effective multimodal (text, image, video) |
| **Amazon Nova** | Nova Pro | `us.amazon.nova-pro-v1:0` | High-capability reasoning, complex instructions |
| **Amazon Titan** | Titan Text Express | `amazon.titan-text-express-v1` | General text generation, chat, summarization |
| **Amazon Titan** | Titan Text Premier | `amazon.titan-text-premier-v1:0` | Enterprise advanced text generation and RAG |
| **Amazon Titan** | Titan Image Generator v2 | `amazon.titan-image-generator-v2:0` | High-resolution image generation and editing |


## 1. Setup & Client Initialization

Interaction with Amazon Bedrock uses two Boto3 clients:
1. `bedrock`: The control plane client (listing models, checking access, managing guardrails/profiles).
2. `bedrock-runtime`: The data plane client (running inference via `invoke_model`).


In [ ]:
import boto3
import json
import os

# Configure target region (e.g. us-east-1 or us-east-2)
REGION = "us-east-2"

bedrock_ctrl = boto3.client(service_name="bedrock", region_name=REGION)
bedrock_runtime = boto3.client(service_name="bedrock-runtime", region_name=REGION)

print(f"Bedrock clients successfully initialized for region: {REGION}")


### Discover Available Amazon Foundation Models

We can check the connection and list all Amazon-provided foundation models in our region with `list_foundation_models(byProvider="Amazon")`:


In [ ]:
response = bedrock_ctrl.list_foundation_models(byProvider="Amazon")

print(f"{'Model Name':<32} {'Model ID':<40} {'Input Modalities'}")
print("-" * 90)
for m in response.get("modelSummaries", []):
    name = m.get("modelName", "")
    mid = m.get("modelId", "")
    in_mod = ", ".join(m.get("inputModalities", []))
    print(f"{name:<32} {mid:<40} {in_mod}")


## 2. Common Inference Parameters

Amazon foundation models support configuration parameters to control generation behavior:

- **Temperature** (`0.0` - `1.0`): Controls randomness. Lower values (e.g., `0.1`) produce more predictable, factual text; higher values (e.g., `0.8`) encourage creative variance.
- **Top P** (`0.0` - `1.0`): Nucleus sampling. The model cuts off candidate tokens whose cumulative probability exceeds `Top P`.
- **Top K** (integer): Restricts selection to the `K` most probable next tokens at each step.
- **Max Tokens** (`max_new_tokens` / `maxTokenCount`): Maximum number of output tokens generated before stopping.
- **Stop Sequences**: Specific character sequences that immediately halt output when generated.


## 3. Invoking Amazon Nova Models

Amazon Nova models use the unified **`messages-v1`** payload schema. To invoke Nova in Bedrock, pass either a system inference profile ID (e.g. `us.amazon.nova-lite-v1:0` or `us.amazon.nova-micro-v1:0`) or the model ID:


In [ ]:
prompt = "Explain the advantages of serverless computing for cloud applications in 3 bullet points."

nova_payload = {
    "schemaVersion": "messages-v1",
    "messages": [
        {
            "role": "user",
            "content": [{"text": prompt}]
        }
    ],
    "inferenceConfig": {
        "max_new_tokens": 350,
        "temperature": 0.7,
        "top_p": 0.9,
        "top_k": 20
    }
}

NOVA_MODEL_ID = "us.amazon.nova-lite-v1:0"  # or us.amazon.nova-micro-v1:0

response = bedrock_runtime.invoke_model(
    modelId=NOVA_MODEL_ID,
    contentType="application/json",
    accept="application/json",
    body=json.dumps(nova_payload)
)

result = json.loads(response["body"].read())
print("--- Amazon Nova Response ---")
print(result["output"]["message"]["content"][0]["text"])
print("\nToken Usage:", result.get("usage"))


## 4. Invoking Amazon Titan Text Models

Amazon Titan Text models (`amazon.titan-text-express-v1` or `amazon.titan-text-premier-v1:0`) use `inputText` and `textGenerationConfig`:


In [ ]:
titan_payload = {
    "inputText": "What are the core differences between relational and non-relational databases?",
    "textGenerationConfig": {
        "maxTokenCount": 300,
        "temperature": 0.5,
        "topP": 0.9,
        "stopSequences": []
    }
}

TITAN_MODEL_ID = "amazon.titan-text-express-v1"

try:
    response = bedrock_runtime.invoke_model(
        modelId=TITAN_MODEL_ID,
        contentType="application/json",
        accept="application/json",
        body=json.dumps(titan_payload)
    )
    
    result = json.loads(response["body"].read())
    print("--- Amazon Titan Text Response ---")
    print(result["results"][0]["outputText"].strip())
except Exception as e:
    print(f"Note: Verify {TITAN_MODEL_ID} is enabled in Bedrock Model Access: {e}")


## 5. Image Generation with Amazon Titan Image Generator

Amazon Titan Image Generator (`amazon.titan-image-generator-v2:0`) generates high-quality images from text prompts.
The API returns a base64 string under `images[0]`, which we decode using `PIL.Image`:


In [ ]:
import base64
import io
from PIL import Image
from IPython.display import display

image_prompt = "A serene mountain lake at sunrise with reflections on the water, high resolution landscape photography"

image_payload = {
    "taskType": "TEXT_IMAGE",
    "textToImageParams": {
        "text": image_prompt
    },
    "imageGenerationConfig": {
        "numberOfImages": 1,
        "quality": "standard",
        "height": 512,
        "width": 512,
        "cfgScale": 8.0,
        "seed": 42
    }
}

IMAGE_MODEL_ID = "amazon.titan-image-generator-v2:0"

try:
    response = bedrock_runtime.invoke_model(
        modelId=IMAGE_MODEL_ID,
        contentType="application/json",
        accept="application/json",
        body=json.dumps(image_payload)
    )
    
    response_body = json.loads(response["body"].read())
    base64_img = response_body["images"][0]
    
    # Decode and display image
    img = Image.open(io.BytesIO(base64.b64decode(base64_img)))
    display(img)
    print(f"Generated image successfully for: '{image_prompt}'")
except Exception as e:
    print(f"Note: Verify {IMAGE_MODEL_ID} access is requested in Bedrock Console: {e}")


## 6. Prompt Engineering with Amazon Models

### 6.1 Zero-Shot Prompting
In zero-shot prompting, the model is given direct instructions without sample input-output pairs:


In [ ]:
zero_shot_prompt = """You are a customer service manager.
Write a professional and polite email to customer \"John Doe\" acknowledging that their support ticket experienced a 2-day delay.
Assure them that their issue is being actively investigated and will be resolved within 24 hours.
"""

payload = {
    "schemaVersion": "messages-v1",
    "messages": [{"role": "user", "content": [{"text": zero_shot_prompt}]}],
    "inferenceConfig": {"max_new_tokens": 350, "temperature": 0.5}
}

response = bedrock_runtime.invoke_model(
    modelId="us.amazon.nova-lite-v1:0",
    contentType="application/json",
    accept="application/json",
    body=json.dumps(payload)
)

result = json.loads(response["body"].read())
print(result["output"]["message"]["content"][0]["text"])


### 6.2 Structured Contextual Prompting (XML Tags)

Using XML-style delimiters (`<customer_feedback>`) helps the model separate instructions from user-provided data:


In [ ]:
structured_prompt = """You are a customer service manager. Draft an email response addressing the customer feedback provided below.

<customer_feedback>
Customer Name: John Doe
Complaint: Waited 3 days for technician callback. Initial troubleshooting steps were incorrect, causing downtime.
Account Tier: Enterprise Gold
</customer_feedback>

Guidelines:
1. Acknowledge the delay and apologize sincerely.
2. Offer a dedicated account engineer and 1-month service credit.
3. Keep the response under 150 words.
"""

payload = {
    "schemaVersion": "messages-v1",
    "messages": [{"role": "user", "content": [{"text": structured_prompt}]}],
    "inferenceConfig": {"max_new_tokens": 350, "temperature": 0.3}
}

response = bedrock_runtime.invoke_model(
    modelId="us.amazon.nova-lite-v1:0",
    contentType="application/json",
    accept="application/json",
    body=json.dumps(payload)
)

result = json.loads(response["body"].read())
print(result["output"]["message"]["content"][0]["text"])


## 7. Clean Up & Resource Management

### Resource Lifecycle & Pricing
- **No Persistent Compute**: On-demand `invoke_model` requests are stateless API calls. They do not provision EC2 instances, SageMaker endpoints, or persistent hourly-billed resources.
- **Zero Idle Cost**: You are charged solely for input and output tokens (or generated images) processed during the call. When not actively executing, ongoing cost is **$0.00**.

### Recommended Cleanup Steps:
1. **Kernel Shutdown**: When done, shut down the Jupyter notebook kernel (`Kernel` -> `Shut Down Kernel`) to release local system RAM.
2. **Local Artifacts**: Delete any test images saved to disk if they are no longer required.
3. **Cost Monitoring**: Verify your token usage in the [AWS Cost Explorer](https://console.aws.amazon.com/cost-management/home) and Amazon CloudWatch Bedrock metrics.

---
### Next Steps
- [002-Streaming-response.ipynb](002-Streaming-response.ipynb): Learn how to stream tokens in real-time using `invoke_model_with_response_stream`.
- [003-Guardrails.ipynb](003-Guardrails.ipynb): Learn how to implement content filters, PII masking, and denied topic safeguards.
